In [2]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import importlib

import models.NN_layers as NN_layers
importlib.reload(NN_layers)
from models.NN_layers import *

import models.Model as Model
importlib.reload(Model)
from models.Model import *

import torch 
import torch.nn as nn
import torch.nn.functional  as F

from torch import Tensor
from torchvision import transforms

import pandas as pd
import PIL.Image as Image

import kagglehub
path = kagglehub.dataset_download("zalando-research/fashionmnist")

c:\Users\katin\OneDrive - Danmarks Tekniske Universitet\6 semester\Fagprojekt\Fagprojekt---Group-Equivariance\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Load billeder

In [3]:
df = pd.read_csv(path + '/fashion-mnist_train.csv') #læs billeder

In [4]:
def format_img(num_images = 1000):
    df_train = df[:num_images]
    img_rows = df_train.iloc[:, 1:].to_numpy() #converts to numpy array (outer dim is pictures)
    img_square = img_rows.reshape(-1, 28,28).astype(np.uint8) # reshape inner dim to be a pictur HxW
    images = torch.tensor(img_square, dtype=torch.complex64).unsqueeze_(1) / 255.0  # make into tensor and scale pixel values to be in range [0,1] instead of [0,255]
    return images

In [5]:
images = format_img(num_images=3)

### Roter billeder til 8 vinkler

In [6]:
def rotate_batch(images, angles_deg):
    images = images.real.to(torch.float64)
    # images: [B, C, H, W]
    B, C, H, W = images.shape
    device = images.device

    out = []

    for angle in angles_deg:
        theta = np.radians(angle)

        # rotation matrix (inverse mapping for grid_sample)
        rot = torch.tensor([
            [np.cos(theta), -np.sin(theta), 0],
            [np.sin(theta),  np.cos(theta), 0]
        ], device=device).unsqueeze(0).repeat(B, 1, 1)

        grid = F.affine_grid(rot, images.size(), align_corners=False)
        rotated = F.grid_sample(images, grid, align_corners=False)
        out.append(rotated)
    return torch.stack(out, dim=1).to(torch.complex64)  # [B, 8, C, H, W]

In [7]:
angles = [i * 45 for i in range(8)]

rotated_images = rotate_batch(images, angles)
print(rotated_images.shape)

torch.Size([3, 8, 1, 28, 28])


In [ ]:
rotated_images[0,0].shape

### Compute output from model

In [8]:
os.getcwd()

'c:\\Users\\katin\\OneDrive - Danmarks Tekniske Universitet\\6 semester'